# CheckThat 2025 — Reasoning Trace Generation (Qwen3-8B, HF pipeline, local RTX 2080 Ti)

Goal: per (claim, evidences) pair, gen 20 reasoning traces at 20 diff fixed temps.
Few-shot prompted. Output label ∈ {True, False, Conflicting}.

**Adapted for local PC** (single NVIDIA RTX 2080 Ti, 11GB VRAM, i5-14600K, 64GB RAM,
Ubuntu 24.04) — originally built for Kaggle's 2×T4 (32GB total). Generation logic,
prompt, temperature schedule, and label handling are unchanged from the original.
What changed is only what had to change to run well on this single-GPU, 11GB-VRAM,
Turing-architecture box:

- `DEVICE_MAP` targets a single GPU instead of spreading layers across two T4s.
- Attention implementation is pinned to `sdpa` (PyTorch's built-in scaled-dot-product
  attention). Turing GPUs (2080 Ti, T4) do **not** support FlashAttention-2 — it
  requires Ampere or newer — so letting `transformers` auto-pick it would error out
  or silently fall back anyway. `sdpa` is fast and ships with PyTorch, no extra
  compile step needed.
- 4-bit NF4 quantization (bitsandbytes) is kept — it's the same config as the
  original, and is *more* necessary here since one 2080 Ti has 11GB vs. 15GB on a
  single T4.
- `bnb_4bit_compute_dtype` stays `float16` — Turing has no native BF16 tensor core
  support, so FP16 is the correct (and originally-used) choice; no change needed there.
- Kaggle-specific paths (`/kaggle/input/...`, `/kaggle/working/...`) are swapped for
  local filesystem paths you set below.
- The Kaggle "restart session" step is removed — that was only needed because
  Kaggle's kernel process had already imported a stale `transformers` into memory
  before the pip upgrade ran. In a fresh local venv this doesn't apply; just install
  once, then run all cells top to bottom.
- Added a couple of small local-environment conveniences (VRAM check, output
  directory creation) that don't touch any generation logic.


In [ ]:
# Environment setup (run once). Recommended: do this inside a virtualenv/conda env.
#
# Same root cause as on Kaggle: Qwen3Config's @strict decorator needs a transformers
# version released together with a compatible huggingface_hub. We pin both together.
# On a fresh local venv there's no stale-import issue like Kaggle has, so no restart
# is required after this — just run the notebook top to bottom.
!pip install -q -U "transformers>=4.57" "huggingface_hub>=0.35" "tokenizers>=0.22" "accelerate>=1.11" "bitsandbytes>=0.48"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# sanity check versions
import transformers, huggingface_hub, tokenizers, accelerate
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)

# quick import test — this line alone reproduces the bug if versions are still stale
from transformers import Qwen3Config
print("Qwen3Config imported OK")


In [ ]:
import json, os, re, time, gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen3-8B"
DEVICE_MAP = "auto"  # single RTX 2080 Ti (11GB) — "auto" just maps everything onto the one GPU it finds

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM: {total_vram_gb:.1f} GB")


## Load model

Qwen3-8B is ungated — no HF token needed. Instruct-tuned, has chat template + built-in
"thinking mode" (emits `<think>...</think>` block before answer).

4-bit NF4 quant via bitsandbytes, same as the original — needed even more here since
we have a single 11GB card instead of 2×15GB. `attn_implementation="sdpa"` is pinned
explicitly because the 2080 Ti's Turing architecture (compute capability 7.5) doesn't
support FlashAttention-2 (that needs Ampere/8.0+); `sdpa` is PyTorch's native fused
attention kernel and works well on Turing without any extra install.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP,
    attn_implementation="sdpa",  # Turing (2080 Ti) has no FlashAttention-2 support; sdpa is the right fit
)
model.eval()
print("Model loaded. Device map:", model.hf_device_map)

if torch.cuda.is_available():
    allocated_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)
    print(f"VRAM allocated: {allocated_gb:.2f} GB | reserved: {reserved_gb:.2f} GB")


## Few-shot prompt template

System + few-shot examples (from CheckThat annotated samples) + target claim/evidence.
Model must output free-text justification ending in `Label: True|False|Conflicting`.


In [ ]:
SYSTEM_PROMPT = (
    "You are a numerical fact-checking assistant for the CLEF CheckThat! 2025 claim verification task. "
    "Given a CLAIM and a set of EVIDENCE snippets, write a short justification reasoning about "
    "whether the evidence supports, contradicts, or gives conflicting signals about the claim. "
    "End your answer with a single line in the exact format: 'Label: True' or 'Label: False' or "
    "'Label: Conflicting'. Do not add anything after the Label line."
)

FEWSHOT_EXAMPLES = [
    {
        "claim": "Six masks worn to school by children were sent to a University of Florida laboratory and found to carry at least 11 dangerous pathogens.",
        "evidences": [
            "15 juin 2021  the resulting report found that five masks were contaminated with bacteria, parasites, and fungi, including three with dangerous pathogenic and ...",
            "in june 2021, a research facility at the university of florida tested six masks worn by children during their school day and found that at least 11 disease-causing pathogens were present on the mask material.",
            "gainesville parents in florida concerned about the harm caused to their children wearing face masks all day at school in 90 f weather sent out six masksfive that were worn by children ages 6 to 11 for five to eight hours at school, and one worn by an adultto be analyzed for contaminants at the university of floridas mass spectrometry research and education center. of the six masks, three were surgical, two cotton, and a poly gaiter. masks that have not been worn and a t-shirt worn at school acted as the control samples. a laboratory at the university of florida that recently analyzed a small sample of face masks, detected the presence of 11 dangerous pathogens that included bacterias that cause diphtheria, pneumonia, and meningitis."
        ],
        "reasoning_trace": (
            "Justification: The claim states that six masks worn by children were tested and found to "
            "carry at least 11 dangerous pathogens. The evidence confirms this, stating that a research "
            "facility at the University of Florida tested six masks worn by children and found that there "
            "were indeed 11 disease-causing pathogens present. Therefore, the numerical span in the claim "
            "matches with the evidence provided.\n\nLabel: True"
        ),
    },
    {
        "claim": "Six masks worn to school by children were sent to a University of Florida laboratory and found to carry at least 11 dangerous pathogens.",
        "evidences": [
            "15 juin 2021  the resulting report found that five masks were contaminated with bacteria, parasites, and fungi, including three with dangerous pathogenic and ...",
            "in june 2021, a research facility at the university of florida tested six masks worn by children during their school day and found that at least 11 disease-causing pathogens were present on the mask material.",
            "gainesville parents in florida concerned about the harm caused to their children wearing face masks all day at school in 90 f weather sent out six masksfive that were worn by children ages 6 to 11 for five to eight hours at school, and one worn by an adultto be analyzed for contaminants at the university of floridas mass spectrometry research and education center. of the six masks, three were surgical, two cotton, and a poly gaiter. masks that have not been worn and a t-shirt worn at school acted as the control samples. a laboratory at the university of florida that recently analyzed a small sample of face masks, detected the presence of 11 dangerous pathogens that included bacterias that cause diphtheria, pneumonia, and meningitis."
        ],
        "reasoning_trace": (
            "Justification: The evidence clearly supports the claim that masks worn by children were "
            "tested and found to carry at least 11 dangerous pathogens. It mentions that six masks were "
            "analyzed, confirming the presence of these pathogens as stated in the claim.\n\nLabel: True"
        ),
    },
]

def build_messages(claim, evidences):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEWSHOT_EXAMPLES:
        user_turn = "CLAIM: " + ex["claim"] + "\n\nEVIDENCE:\n" + "\n".join(
            f"- {e}" for e in ex["evidences"]
        )
        msgs.append({"role": "user", "content": user_turn})
        msgs.append({"role": "assistant", "content": ex["reasoning_trace"]})
    target_turn = "CLAIM: " + claim + "\n\nEVIDENCE:\n" + "\n".join(f"- {e}" for e in evidences)
    msgs.append({"role": "user", "content": target_turn})
    return msgs


## Temperature schedule (20 fixed values, 0.0 → 1.0)

Even spread across [0.0, 1.0], 20 steps. temp=0.0 -> greedy (near-deterministic) baseline trace,
rest ramp up sampling diversity linearly.


In [ ]:
import numpy as np

TEMPERATURES = [float(t) for t in np.round(np.linspace(0.0, 1.0, 20), 3)]
assert len(TEMPERATURES) == 20
print(TEMPERATURES)


## Label extraction + generation fn

In [ ]:
LABEL_RE = re.compile(r"Label:\s*(True|False|Conflicting)", re.IGNORECASE)

def extract_label(text):
    m = LABEL_RE.search(text)
    if m:
        return m.group(1).capitalize() if m.group(1).lower() != "conflicting" else "Conflicting"
    return None

def normalize_label(raw):
    if raw is None:
        return None
    r = raw.strip().lower()
    if r == "true":
        return "True"
    if r == "false":
        return "False"
    if r.startswith("conflict"):
        return "Conflicting"
    return None

@torch.inference_mode()
def generate_trace(claim, evidences, temperature, max_new_tokens=2500, seed=None):
    messages = build_messages(claim, evidences)
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=True,  # Qwen3: skip <think> block, direct justification only
        return_tensors="pt",
        return_dict=True,  # returns BatchEncoding {input_ids, attention_mask}, not a bare tensor
    ).to(model.device)

    if seed is not None:
        torch.manual_seed(seed)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=max(temperature, 0.01),
        top_p=0.95,
        top_k=20,
        pad_token_id=tokenizer.pad_token_id,
    )
    out = model.generate(**inputs, **gen_kwargs)
    input_len = inputs["input_ids"].shape[-1]
    new_tokens = out[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    label = normalize_label(extract_label(text))
    return text, label


## Input data

Load input JSON in the agreed schema:
```json
[{"claim": "...", "original_id": 1, "number_of_unique_docs": 3, "docs": ["...", "...", "..."]}, ...]
```
Adjust `INPUT_PATH` / `OUTPUT_PATH` below to wherever the dataset lives on your PC.


In [ ]:
INPUT_PATH = "./data/3-processed-reranked.json"      # <-- adjust to your local dataset path
OUTPUT_PATH = "./output/reasoning_traces.json"       # <-- adjust to where you want results saved

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    all_claims = json.load(f)

print(f"Total claims in input: {len(all_claims)}")

SUBSET = all_claims
print(f"Processing subset: {len(SUBSET)} claims")


## Run generation loop

20 traces per claim, one per temperature. Iterates **last claim to first**. Saves incrementally after each claim (crash-safe). Skips any claim whose `original_id` is already present in `OUTPUT_PATH` (resume behavior) or, if set, in `EXTERNAL_DONE_PATH` (a separate file of already-generated traces you want to skip, e.g. from another run).


In [ ]:
# Optional: path to a separate file that already has reasoning traces generated
# (e.g. from a prior/partial run, or someone else's output) — claims found here are
# skipped too, in addition to whatever's already in OUTPUT_PATH. Set to None to disable.
EXTERNAL_DONE_PATH = None  # <-- e.g. "./output/reasoning_traces_from_other_run.json"

results = []

if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        results = json.load(f)
    done_ids = {r["original_id"] for r in results}
else:
    done_ids = set()

if EXTERNAL_DONE_PATH and os.path.exists(EXTERNAL_DONE_PATH):
    with open(EXTERNAL_DONE_PATH, "r", encoding="utf-8") as f:
        external_results = json.load(f)
    external_ids = {r["original_id"] for r in external_results}
    print(f"Loaded {len(external_ids)} already-done claim IDs from {EXTERNAL_DONE_PATH}")
    done_ids |= external_ids

# process last claim to first
for item in reversed(SUBSET):
    if item["original_id"] in done_ids:
        print(f"Skip {item['original_id']} (already done)")
        continue

    claim = item["claim"]
    docs = item["docs"]
    print(f"\n=== Claim {item['original_id']}: {claim[:80]}... ===")

    traces = []
    for i, temp in enumerate(TEMPERATURES, start=1):
        t0 = time.time()
        text, label = generate_trace(claim, docs, temperature=temp, seed=1000 * item["original_id"] + i)
        dt = time.time() - t0
        traces.append({
            "trace_id": i,
            "temperature": temp,
            "label": label,
            "reasoning_trace": text,
        })
        print(f"  trace {i:2d}  T={temp:.2f}  label={label}  ({dt:.1f}s)")

    results.append({
        "claim": claim,
        "original_id": item["original_id"],
        "reasoning_traces": traces,
    })

    # incremental save
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    gc.collect()
    torch.cuda.empty_cache()

print("\nDone. Saved to", OUTPUT_PATH)


## Quick QA view

In [ ]:
for r in results:
    print("="*100)
    print("Claim:", r["claim"])
    print("ID:", r["original_id"])
    label_counts = {}
    for t in r["reasoning_traces"]:
        label_counts[t["label"]] = label_counts.get(t["label"], 0) + 1
    print("Label distribution across 20 traces:", label_counts)
